# S02 — Kinematic self-consistency

Track: GT-free raw hand fixtures. Split: development only. Fitting, scoring and calibration await a separately invoked S03.

The raw residual is $r=\|(p_1-p_0)-R_0v_0\Delta t\|_2$ in **metres**, using the latest two causal samples. Position is in receiver metres; prior velocity is in body m/s unless explicitly marked receiver m/s. R maps body to receiver. Samples must have source time at/before the anchor and receipt by the deadline. The prior velocity alone predicts displacement; the current velocity cannot repair the residual.

Hand comparisons: 2 m/s × 0.5 s predicts 1 m. Legitimate acceleration from 2 to 3 m/s over 0.5 s moves 1.25 m, producing 0.25 m residual. The inconsistent report moves 3 m, producing 2 m residual. The jointly spoofed pair reports 100→103 m with 6 m/s, producing zero residual. These are explicit hand fixtures, not evidence of detection performance.

Pass criterion: all hand expectations and missing/invalid/causal states agree; a nonzero residual is not an accusation. Regions are **not applicable** to this factor; no evaluator GT or independent motion measurement is consumed.

In [ ]:
import json, os, sys
from pathlib import Path
from IPython.display import display, Markdown, Image, JSON
from valid_point.evidence import load_raw_config, export_demo
from valid_point.provenance import sha256_file
root = Path(os.environ['VP_REPO_ROOT'])
run_dir = Path(os.environ['VP_RUN_DIR'])
config = load_raw_config(run_dir / 'config.json')
provenance = dict(run_id=run_dir.name, stage='S02', track=config['track'],
                  segment=config['segment'], config_sha256=sha256_file(run_dir/'config.json'),
                  source_sha256=os.environ['VP_SOURCE_SHA256'], notebook_section='02_kinematics')
print('Fresh kernel:', sys.executable)
print(json.dumps(provenance, indent=2))
display(JSON({k:v for k,v in config.items() if not k.endswith('_cases')}, expanded=True))
print('Reference/calibration: not fitted. Normalized evidence, score, alarm, allocation: not run.')

## Inputs and source provenance

Every literal input is retained below and as a hashed run-local JSON file. Region authority, bounds and freeze time are in the config; kinematics/availability do not use regions. Sender report IDs and receiver context/transform provenance remain separate.

In [ ]:
evidence = export_demo('kinematics', root, run_dir, config, provenance)
display(JSON(evidence['inputs'], expanded=True))

## Raw outputs and unknown states

The full table includes all cases, failures of measurement eligibility, null values and reasons. No normalized evidence is available in S02.

In [ ]:
display(Markdown(evidence['tables'][1].read_text()))
display(Image(filename=str(evidence['figures'][2])))

## Hand expectations, tests and gate

In [ ]:
display(Markdown(evidence['check_tables'][1].read_text()))
tests = json.loads((run_dir/'test_results.json').read_text())
print(tests['output'])
print('Notebook raw-contract gate:', evidence['status'])
print('Run test gate:', 'PASS' if tests['exit_code'] == 0 else 'FAIL')
assert evidence['status'] == 'PASS'
assert tests['exit_code'] == 0

## Conclusion and claim limits

In [ ]:
display(Markdown(evidence['limitations']))
print('S02 raw-contract gate:', evidence['status'], '; G2 score/calibration gate: BLOCKED pending uninvoked S03')
for path in evidence['tables'] + evidence['check_tables'] + evidence['figures']:
    print(path.relative_to(root), sha256_file(path))